# Lecture 5: Building a Panel Dataset 
**BANA 4373 — Applied Data Tools for Economics & Business**  
**Dr. Fidel González — Spring 2026**

This notebook is designed to run *seamlessly in class* **without any external downloads**.  
We will simulate a realistic “raw” workflow using county-level unemployment data across multiple years, then:

- Clean identifiers (FIPS as strings, preserve leading zeros)
- Stack years to form a county–year panel
- Diagnose duplicates and coverage gaps
- Decide how to handle common panel issues
- Create a mini data dictionary
- Export a cleaned panel dataset


---
## 0) Learning Objectives
By the end of this notebook, you can:

1. Define the **panel key** and verify it uniquely identifies rows.
2. Construct a county–year panel by stacking yearly files.
3. Run **panel diagnostics** (duplicates, gaps, coverage changes).
4. Document a cleaned dataset in a mini data dictionary.

> **Structure first. Model second.**


In [ ]:
# 1) Setup
import os
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

ROOT = Path.cwd() / "lecture5_panel_demo"
RAW_DIR = ROOT / "data_raw"
CLEAN_DIR = ROOT / "data_clean"
EXPORT_DIR = ROOT / "exports"

for d in [RAW_DIR, CLEAN_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ROOT, RAW_DIR, CLEAN_DIR, EXPORT_DIR


---
## 2) Create “Raw” Yearly Files (Simulated)
In real work, you would download BLS LAUS county files. In class (and in this demo notebook), we create **raw CSVs** that intentionally include common issues:

- FIPS sometimes numeric (leading zeros can be lost)
- County names not standardized
- Duplicate county–year rows (a classic merge/panel pitfall)
- A county missing in one year (unbalanced panel)

This makes the workflow realistic and reproducible.


In [ ]:
# 2) Build small raw datasets with realistic issues
# We'll simulate 3 years: 2019–2021
# Columns mimic a typical county-level economic file

raw_2019 = pd.DataFrame({
    "state_fips": [48, 48, 48, 48, 48],
    "county_fips": [201, 157, 339, 291, 201],  # note duplicate key (48,201) appears twice in 2019
    "county_name": ["Harris County", "Fort Bend County", "Montgomery County", "Liberty County", "HARRIS COUNTY"],
    "unemp_rate": [3.6, 3.2, 3.4, 4.1, 3.6],  # duplicated row repeats value
    "source": ["BLS_LAUS"] * 5
})

raw_2020 = pd.DataFrame({
    "state_fips": [48, 48, 48, 48],
    "county_fips": [201, 157, 339, 291],
    "county_name": ["Harris County", "Fort Bend County", "Montgomery Cnty", "Liberty County"],  # inconsistent name
    "unemp_rate": [8.1, 7.4, 7.9, 9.2],
    "source": ["BLS_LAUS"] * 4
})

raw_2021 = pd.DataFrame({
    # Here we intentionally store FIPS as strings already, including leading zeros (none here, but pattern matters)
    "state_fips": ["48", "48", "48"],
    "county_fips": ["201", "157", "339"],
    "county_name": ["Harris County", "Fort Bend County", "Montgomery County"],
    "unemp_rate": [5.3, 4.9, 5.1],
    "source": ["BLS_LAUS"] * 3
    # Note: Liberty County (291) is missing in 2021 → unbalanced panel
})

# Write to CSV as "raw files"
raw_2019.to_csv(RAW_DIR / "laus_county_2019_raw.csv", index=False)
raw_2020.to_csv(RAW_DIR / "laus_county_2020_raw.csv", index=False)
raw_2021.to_csv(RAW_DIR / "laus_county_2021_raw.csv", index=False)

sorted([p.name for p in RAW_DIR.glob("*.csv")])


---
## 3) Load One Year and Validate the Key
**Rule from Lecture 4 → Lecture 5:** Never start building a panel until you can clearly state what uniquely identifies a row.

For a county-year panel, the key is:

- `state_fips` + `county_fips` + `year`

But within a single year file (before adding `year`), the key should be:

- `state_fips` + `county_fips`


In [ ]:
# 3) Load 2019 (one year) as the baseline
df_2019_raw = pd.read_csv(RAW_DIR / "laus_county_2019_raw.csv")
df_2019_raw.head()


In [ ]:
# Check shape and dtypes (notice FIPS likely read as int)
df_2019_raw.shape, df_2019_raw.dtypes


### 3.1) Why FIPS should be stored as strings
Even if a state/county code *looks* numeric, it’s an identifier.

**Best practice:** Store FIPS as strings to preserve leading zeros and avoid accidental numeric operations.


In [ ]:
def clean_fips(df: pd.DataFrame) -> pd.DataFrame:
    """Clean FIPS codes (state_fips, county_fips) as zero-padded strings.
    - state_fips: 2 digits
    - county_fips: 3 digits
    """
    out = df.copy()
    out["state_fips"] = out["state_fips"].astype(str).str.zfill(2)
    out["county_fips"] = out["county_fips"].astype(str).str.zfill(3)
    return out

df_2019 = clean_fips(df_2019_raw)
df_2019.dtypes, df_2019.head()


In [ ]:
# 3.2) Define the within-year key and test uniqueness
within_year_key = ["state_fips", "county_fips"]

dup_mask_2019 = df_2019.duplicated(subset=within_year_key, keep=False)
df_2019.loc[dup_mask_2019].sort_values(within_year_key)


✅ We found a duplicate key in 2019.

In practice, duplicates happen because:
- A data provider accidentally repeats rows
- Two records represent different subcategories but you didn't filter
- You merged incorrectly earlier

**Before building a panel, you must decide how to handle duplicates.**


In [ ]:
# 3.3) A simple duplicate-handling strategy for this example:
# If duplicates exist with the exact same values, we can drop duplicates safely.
# (In real work, you'd investigate why duplicates exist.)
df_2019_dedup = df_2019.drop_duplicates(subset=within_year_key, keep="first").copy()

df_2019.shape, df_2019_dedup.shape


---
## 4) Load Multiple Years and Stack to Build the Panel
Panels grow **vertically** (append/concat). We will:

1. Load each year file
2. Clean identifiers
3. Add a `year` column
4. Stack into a single dataframe


In [ ]:
def load_and_prepare_year(year: int) -> pd.DataFrame:
    """Load a raw year file, clean identifiers, add year, standardize columns."""
    path = RAW_DIR / f"laus_county_{year}_raw.csv"
    df = pd.read_csv(path)
    df = clean_fips(df)
    df["year"] = int(year)

    # Basic name standardization (illustrative)
    df["county_name"] = (
        df["county_name"]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Standardize common abbreviations (demo)
    df["county_name"] = df["county_name"].replace({
        "Montgomery Cnty": "Montgomery County",
        "HARRIS COUNTY": "Harris County"
    })

    # Ensure expected columns exist
    expected = ["state_fips", "county_fips", "county_name", "unemp_rate", "source", "year"]
    df = df[expected]
    return df

years = [2019, 2020, 2021]
dfs = [load_and_prepare_year(y) for y in years]

for y, d in zip(years, dfs):
    print(y, d.shape, d.dtypes.to_dict())


In [ ]:
# Handle duplicates within each year BEFORE stacking (important professional habit)
def dedup_within_year(df: pd.DataFrame) -> pd.DataFrame:
    key = ["state_fips", "county_fips", "year"]
    # If duplicates exist, keep first and print a warning
    n_dups = df.duplicated(subset=key).sum()
    if n_dups > 0:
        print(f"WARNING: Found {n_dups} duplicate county-year rows. Dropping duplicates (keep first).")
    return df.drop_duplicates(subset=key, keep="first").copy()

dfs_dedup = [dedup_within_year(d) for d in dfs]

panel_raw = pd.concat(dfs_dedup, ignore_index=True)
panel_raw.head(), panel_raw.shape


---
## 5) Panel Diagnostics
After stacking, always answer:

1. **Are keys unique?** (no duplicate unit–time rows)
2. **Is the panel balanced or unbalanced?**
3. **Are there coverage gaps?** (units missing in certain years)

Panel key (county–year):
- `state_fips` + `county_fips` + `year`


In [ ]:
panel_key = ["state_fips", "county_fips", "year"]

# 5.1) Key uniqueness
n_dups_panel = panel_raw.duplicated(subset=panel_key).sum()
n_dups_panel


In [ ]:
# 5.2) Coverage by year (how many counties each year?)
coverage_by_year = panel_raw.groupby("year")[["county_fips"]].count().rename(columns={"county_fips":"n_counties"})
coverage_by_year


In [ ]:
# 5.3) Determine balance: do all counties appear in all years?
units = panel_raw[["state_fips", "county_fips"]].drop_duplicates()
n_units = len(units)
n_years = panel_raw["year"].nunique()

# Count how many years each unit appears
years_per_unit = (panel_raw.groupby(["state_fips", "county_fips"])["year"]
                  .nunique()
                  .reset_index(name="years_observed"))

years_per_unit.sort_values("years_observed").head(10), n_units, n_years


In [ ]:
# Identify missing unit-years explicitly
# Build the full expected grid of (unit x year) and compare
full_grid = units.assign(key=1).merge(pd.DataFrame({"year": sorted(panel_raw["year"].unique()), "key": 1}), on="key").drop(columns="key")

panel_present = panel_raw[panel_key].drop_duplicates()
missing_unit_years = full_grid.merge(panel_present, on=panel_key, how="left", indicator=True)
missing_unit_years = missing_unit_years[missing_unit_years["_merge"] == "left_only"].drop(columns="_merge")

missing_unit_years


✅ Interpretation:

- If `missing_unit_years` is non-empty, the panel is **unbalanced**.
- Unbalanced panels are common in applied economics.
- What matters is that you **know** and **document** the pattern.


---
## 6) Decide on a “Clean” Panel for Analysis
In real work, you have choices:

- Keep an unbalanced panel (most common)
- Restrict to a balanced panel (can drop lots of data)
- Impute missing values (usually needs a defensible method)

For this lecture, we will create **two versions**:

1. **Unbalanced panel** (keep all observed county-years)
2. **Balanced subset** (keep only counties observed in all years)


In [ ]:
# 6.1) Unbalanced panel (cleaned)
panel_unbalanced = panel_raw.sort_values(panel_key).reset_index(drop=True)

panel_unbalanced.head(), panel_unbalanced.shape


In [ ]:
# 6.2) Balanced subset: keep counties observed in all years
balanced_units = years_per_unit.loc[years_per_unit["years_observed"] == n_years, ["state_fips", "county_fips"]]
panel_balanced = panel_unbalanced.merge(balanced_units, on=["state_fips", "county_fips"], how="inner")

panel_balanced.shape, panel_balanced.groupby("year").size()


---
## 7) Quick QA Summary Tables (No Plots)
These tables help you sanity-check your panel before doing any analysis.


In [ ]:
# 7.1) Summary stats by year
summary_by_year = (panel_unbalanced
                   .groupby("year")["unemp_rate"]
                   .agg(["count","mean","min","max"])
                   .round(2))

summary_by_year


In [ ]:
# 7.2) Identify counties with missing years (if any)
missing_by_unit = missing_unit_years.groupby(["state_fips", "county_fips"]).size().reset_index(name="n_missing_years")
missing_by_unit


---
## 8) Mini Data Dictionary (Documentation)
A dataset without documentation is not reusable.

Below is a simple, practical format. In real projects, you would also include:
- Units and transformations
- Any filtering rules
- Known limitations


In [ ]:
data_dictionary = pd.DataFrame([
    {"variable":"state_fips", "description":"2-digit state FIPS code (string, zero-padded)", "units":"ID code", "source":"BLS LAUS (simulated)", "notes":"Stored as string to preserve leading zeros"},
    {"variable":"county_fips", "description":"3-digit county FIPS code within state (string, zero-padded)", "units":"ID code", "source":"BLS LAUS (simulated)", "notes":"Stored as string to preserve leading zeros"},
    {"variable":"county_name", "description":"County name (standardized)", "units":"text", "source":"BLS LAUS (simulated)", "notes":"Minor standardization (case, abbreviations)"},
    {"variable":"year", "description":"Calendar year", "units":"year", "source":"Constructed", "notes":"Explicitly added during panel construction"},
    {"variable":"unemp_rate", "description":"Annual unemployment rate", "units":"percent", "source":"BLS LAUS (simulated)", "notes":"Numeric; check for missingness after merges/stacks"},
    {"variable":"source", "description":"Data source label", "units":"text", "source":"Constructed", "notes":"Useful for provenance when combining sources"}
])

data_dictionary


---
## 9) Export Cleaned Data
We export:
- `panel_unbalanced.csv`
- `panel_balanced.csv`
- `data_dictionary.csv`

This mirrors a professional workflow: raw → clean → export.


In [ ]:
# Save cleaned datasets
panel_unbalanced_path = EXPORT_DIR / "panel_unbalanced.csv"
panel_balanced_path = EXPORT_DIR / "panel_balanced.csv"
dictionary_path = EXPORT_DIR / "data_dictionary.csv"

panel_unbalanced.to_csv(panel_unbalanced_path, index=False)
panel_balanced.to_csv(panel_balanced_path, index=False)
data_dictionary.to_csv(dictionary_path, index=False)

panel_unbalanced_path, panel_balanced_path, dictionary_path


---
## 10) Final Takeaways
- Panels are **constructed**, not downloaded.
- Your panel key defines whether your dataset is logically valid.
- Always diagnose: duplicates, missing unit-years, coverage changes.
- Unbalanced panels are normal — undocumented panels are not.

Next week (Visualization), we will use this cleaned panel to tell a clear story with figures.
